<a href="https://colab.research.google.com/github/Mknotfound/ML-Final-Lab-Group-04/blob/main/notebooks/Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import os
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, roc_auc_score, confusion_matrix

# 1. Fetch data directly from your repo's raw GitHub URL
DATA_URL = "https://raw.githubusercontent.com/Mknotfound/ML-Final-Lab-Group-04/main/data/raw/sample%20(2).csv"

print("[+] Loading raw dataset from GitHub...")
df = pd.read_csv(DATA_URL)

# 2. Extract features and target labels
X = df.drop(columns=['label'])
y = df['label']

# Filter out unlabeled samples (-1) if present
valid_mask = y != -1
X, y = X[valid_mask], y[valid_mask]

# 3. Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"[+] Train shape: {X_train.shape}, Test shape: {X_test.shape}\n")

# 4. Train LightGBM Baseline Model
print("=== Training LightGBM Baseline Model ===")
lgb_model = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, random_state=42)
lgb_model.fit(X_train, y_train)

lgb_preds = lgb_model.predict(X_test)
lgb_probs = lgb_model.predict_proba(X_test)[:, 1]

# 5. Evaluate Metrics
recall = recall_score(y_test, lgb_preds)
roc_auc = roc_auc_score(y_test, lgb_probs)

print(f"\nMalware Recall: {recall:.4f}")
print(f"ROC-AUC Score:  {roc_auc:.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, lgb_preds))

# 6. Export serialized model artifact locally inside Colab session
os.makedirs('data/processed', exist_ok=True)
model_path = 'data/processed/baseline_model.pkl'
joblib.dump(lgb_model, model_path)

print(f"\n[+] Successfully saved model artifact to '{model_path}'")

[+] Loading raw dataset from GitHub...
[+] Train shape: (1202, 512), Test shape: (301, 512)

=== Training LightGBM Baseline Model ===
[LightGBM] [Info] Number of positive: 625, number of negative: 577
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.010699 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 114970
[LightGBM] [Info] Number of data points in the train set: 1202, number of used features: 512
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.519967 -> initscore=0.079909
[LightGBM] [Info] Start training from score 0.079909

Malware Recall: 0.8535
ROC-AUC Score:  0.9371

Confusion Matrix:
[[123  21]
 [ 23 134]]

[+] Successfully saved model artifact to 'data/processed/baseline_model.pkl'
